# EDA — SPIELPLATZPUNKTOGD.csv (Spielplätze / playgrounds)

City of Vienna open data export. Source file: `data/raw/SPIELPLATZPUNKTOGD.csv`. Run all cells to
reproduce the findings below.

In [1]:
import pandas as pd
import re

df = pd.read_csv("../data/raw/SPIELPLATZPUNKTOGD.csv")
df.shape

(771, 8)

## Columns & dtypes

In [2]:
df.dtypes

FID                      str
OBJECTID               int64
SHAPE                    str
ANL_NAME                 str
BEZIRK               float64
SPIELPLATZ_DETAIL        str
TYP_DETAIL               str
SE_ANNO_CAD_DATA     float64
dtype: object

## Missing values

In [3]:
df.isnull().sum()

FID                    0
OBJECTID               0
SHAPE                  0
ANL_NAME               0
BEZIRK                 5
SPIELPLATZ_DETAIL      0
TYP_DETAIL             0
SE_ANNO_CAD_DATA     771
dtype: int64

## Duplicate check

In [4]:
print("Duplicate full rows:", df.duplicated().sum())
print("Duplicate ANL_NAME:", df["ANL_NAME"].duplicated().sum())

Duplicate full rows: 0
Duplicate ANL_NAME: 137


## Parse geometry

`SHAPE` is a WKT geometry string. Parse into `lon`/`lat` (works for the first
coordinate pair even if the geometry is a line/polygon) and sanity-check the
range against Vienna's bounding box.

In [5]:
def parse_first_point(s):
    m = re.search(r"(-?\d+\.\d+)\s+(-?\d+\.\d+)", str(s))
    if m:
        return float(m.group(1)), float(m.group(2))
    return None, None

df["lon"], df["lat"] = zip(*df["SHAPE"].map(parse_first_point))
print("Unparseable SHAPE values:", df["lon"].isnull().sum())
print("lon range:", df["lon"].min(), "-", df["lon"].max())
print("lat range:", df["lat"].min(), "-", df["lat"].max())

Unparseable SHAPE values: 0
lon range: 16.21250391613166 - 16.534850746055643
lat range: 48.12626393632989 - 48.302619265151776


## District (`BEZIRK`) distribution

In [6]:
df["BEZIRK"].value_counts(dropna=False).sort_index()

BEZIRK
1.0       5
2.0      54
3.0      29
4.0      14
5.0      18
6.0      11
7.0      15
8.0       5
9.0      16
10.0     78
11.0     43
12.0     34
13.0     28
14.0     29
15.0     34
16.0     33
17.0     19
18.0     23
19.0     36
20.0     24
21.0     69
22.0    117
23.0     32
NaN       5
Name: count, dtype: int64

## Investigate the 137 "duplicate" names

Large number of repeats on `ANL_NAME` — check whether these are genuine duplicate
rows or multiple distinct point-features (e.g. separate equipment zones) that
share one playground's name.

In [7]:
dups = df[df.duplicated("ANL_NAME", keep=False)].sort_values("ANL_NAME")
dups[["ANL_NAME", "TYP_DETAIL", "SPIELPLATZ_DETAIL"]].head(10)

,ANL_NAME,TYP_DETAIL,SPIELPLATZ_DETAIL
44,Anna-Freud-Park,"Kleinkinderspielplatz, Spielplatz","Klettern, Rutschen, Sandspiel, Schaukeln, Spie..."
43,Anna-Freud-Park,Ballspielplatz,Basketball
85,Antonspark,"Ballspielplatz, Spielplatz, sonstiger Spielplatz","Balancieren, Klettern, Reck, Rutschen, Sandspi..."
84,Antonspark,"Ballspielkäfig, Ballspielplatz","Basketball, Fußball, Volleyball"
295,Arenbergpark,"Ballspielplatz, Kleinkinderspielplatz, Spielplatz","Basketball, Drehen, Fußball, Klettern, Rutsche..."
294,Arenbergpark,Ballspielplatz,Tischtennis
241,Auer-Welsbach-Park,"Ballspielplatz, sonstiger Spielplatz","Boccia, Slackline"
242,Auer-Welsbach-Park,"Ballspielplatz, Generationenspielplatz, Kleink...","Balancieren, Basketball, Fitness, Klettern, Ru..."
303,Bruno-Kreisky-Park,"Ballspielplatz, Kleinkinderspielplatz, Spielplatz","Drehen, Klettern, Rutschen, Sandspiel, Schauke..."
302,Bruno-Kreisky-Park,Generationenspielplatz,Calisthenics


## Category & equipment detail

`TYP_DETAIL` is a combined category label (comma-separated, e.g. "Ballspielkäfig,
Spielplatz"). `SPIELPLATZ_DETAIL` lists actual equipment (slides, swings, etc.) —
much more granular, potentially very useful for activity-preference matching
("find a playground with a trampoline").

In [8]:
print("Top TYP_DETAIL combinations:")
print(df["TYP_DETAIL"].value_counts().head(10))

print("\nSample SPIELPLATZ_DETAIL (equipment lists):")
for v in df["SPIELPLATZ_DETAIL"].dropna().head(3):
    print("-", v)

Top TYP_DETAIL combinations:
TYP_DETAIL
Spielplatz                                                           142
Kleinkinderspielplatz                                                 76
Ballspielplatz                                                        61
Kleinkinderspielplatz, Spielplatz                                     37
Ballspielplatz, Spielplatz                                            34
Ballspielkäfig                                                        29
Generationenspielplatz                                                25
Ballspielkäfig, Ballspielplatz, Kleinkinderspielplatz, Spielplatz     21
Ballspielplatz, Kleinkinderspielplatz, Spielplatz                     21
Ballspielkäfig, Spielplatz                                            19
Name: count, dtype: int64

Sample SPIELPLATZ_DETAIL (equipment lists):
- Balancieren, Basketball, Fußball, Klettern, Rutschen, Sandspiel, Schaukeln, Tischtennis, Volleyball, Wasserspiel, Wippen
- , Basketball, Fußball, Karussell, 

## Sample rows

In [9]:
df[["ANL_NAME", "BEZIRK", "TYP_DETAIL", "lon", "lat"]].sample(5, random_state=1)

,ANL_NAME,BEZIRK,TYP_DETAIL,lon,lat
286,PA Löwygrube,10.0,"Ballspielplatz, Kleinkinderspielplatz, Spielplatz",16.403510,48.164290
101,PA Küniglberg,13.0,Ballspielplatz,16.291229,48.181194
762,Schweizergarten,3.0,"Kleinkinderspielplatz, sonstiger Spielplatz",16.389108,48.187708
353,PA Wolfersberg,14.0,Spielplatz,16.247338,48.210701
403,Hans-Paulas-Park,11.0,Spielplatz,16.471853,48.159929


## Findings

**Basics:** 771 records, 8 columns — second-largest of the six.

**Missing values:** `BEZIRK` missing for 5 rows only.

**The 137 "duplicate" `ANL_NAME` values are not duplicate POIs** — inspecting them
shows multiple rows sharing a playground name typically have different
`TYP_DETAIL`/`SPIELPLATZ_DETAIL` values, meaning each row is a distinct
point-feature (e.g. a separate ball-cage or toddler area) within the same named
playground, not the same feature recorded twice. Matters for the KG: either model
these as separate `Playground` instances at slightly different coordinates, or
aggregate by `ANL_NAME` into one facility node with multiple sub-features —
worth deciding deliberately rather than deduplicating away real data.

**`SPIELPLATZ_DETAIL` is a genuine asset** — a free-text but consistently
comma-separated equipment list (swings, slides, trampoline, table tennis, water
play, etc.). This is exactly the kind of attribute that supports
interest-based activity filtering in the reasoning layer, more so than most other
files' fields.

**Suitability for the KG:** good candidate, with a genuine open modelling
question (aggregate vs. keep as separate sub-features). Suggested mapping:
`ANL_NAME` → `poi:name`, `BEZIRK` → `poi:district`, `lon`/`lat` →
`geo:long`/`geo:lat`, `TYP_DETAIL` → category tags, `SPIELPLATZ_DETAIL` → parsed
into a list of `poi:hasFeature` values (split on comma) for fine-grained
filtering.